In [0]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import math

In [0]:
class StintLSTM(nn.Module):
    """
    LSTM-based lap time prediction model.
    Uses the SAME dataloader as StintTransformer (dataloader.py).
    
    Key differences from Transformer:
    - Uses LSTM instead of self-attention
    - Processes sequences recurrently (left-to-right)
    - Uses pack_padded_sequence for efficient padding handling
    """
    def __init__(self, n_drivers, n_teams, n_tyres, n_circuits=0,
                 driver_emb_dim=8, team_emb_dim=8, tyre_emb_dim=4, circuit_emb_dim=12,
                 n_cont_features=10, hidden_dim=64, num_layers=2, dropout=0.3):
        super().__init__()
        
        # Embeddings (only create if we have classes to embed)
        self.driver_emb = nn.Embedding(n_drivers, driver_emb_dim) if n_drivers > 0 else None
        self.team_emb = nn.Embedding(n_teams, team_emb_dim) if n_teams > 0 else None
        self.tyre_emb = nn.Embedding(n_tyres, tyre_emb_dim) if n_tyres > 0 else None
        self.circuit_emb = nn.Embedding(n_circuits, circuit_emb_dim) if n_circuits > 0 else None
        
        # Input dimension (adjust based on which embeddings exist)
        actual_driver_dim = driver_emb_dim if n_drivers > 0 else 0
        actual_team_dim = team_emb_dim if n_teams > 0 else 0
        actual_tyre_dim = tyre_emb_dim if n_tyres > 0 else 0
        actual_circuit_dim = circuit_emb_dim if n_circuits > 0 else 0
        input_dim = n_cont_features + actual_driver_dim + actual_team_dim + actual_tyre_dim + actual_circuit_dim
        
        # LSTM layers
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False  # Set to True for BiLSTM
        )
        
        # Output layer
        self.output_layer = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, cont_feats, driver_idx, team_idx, tyre_idx, circuit_idx=None, mask=None):
        """
        Args:
            cont_feats: (batch_size, seq_len, n_cont_features)
            driver_idx: (batch_size,)
            team_idx: (batch_size,)
            tyre_idx: (batch_size, seq_len)
            circuit_idx: (batch_size,) - circuit index
            mask: (batch_size, seq_len) - True for padded positions
        
        Returns:
            out: (batch_size, seq_len) - predicted lap times
        """
        batch_size, seq_len, _ = cont_feats.size()
        
        # Collect embeddings only for features that exist
        embeddings = [cont_feats]
        
        if self.driver_emb is not None:
            driver_emb = self.driver_emb(driver_idx).unsqueeze(1).repeat(1, seq_len, 1)
            embeddings.append(driver_emb)
        
        if self.team_emb is not None:
            team_emb = self.team_emb(team_idx).unsqueeze(1).repeat(1, seq_len, 1)
            embeddings.append(team_emb)
        
        if self.tyre_emb is not None:
            tyre_emb = self.tyre_emb(tyre_idx)
            embeddings.append(tyre_emb)
        
        if self.circuit_emb is not None and circuit_idx is not None:
            circuit_emb = self.circuit_emb(circuit_idx).unsqueeze(1).repeat(1, seq_len, 1)
            embeddings.append(circuit_emb)
        
        # Concatenate all features
        x = torch.cat(embeddings, dim=-1)
        
        # Pack sequences for efficient LSTM processing (optional but faster)
        if mask is not None:
            # Calculate sequence lengths from mask
            seq_lengths = (~mask).sum(dim=1).cpu()  # Count non-padded positions
            
            # Pack padded sequences
            x_packed = pack_padded_sequence(
                x, 
                seq_lengths, 
                batch_first=True, 
                enforce_sorted=False
            )
            
            # LSTM forward pass on packed sequences
            lstm_out_packed, _ = self.lstm(x_packed)
            
            # Unpack sequences
            lstm_out, _ = pad_packed_sequence(lstm_out_packed, batch_first=True)
        else:
            # No masking - process full sequences
            lstm_out, _ = self.lstm(x)
        
        # Apply dropout and output layer
        lstm_out = self.dropout(lstm_out)
        out = self.output_layer(lstm_out).squeeze(-1)  # (batch_size, seq_len)
        
        return out

In [0]:
# ----------------------
# Positional Encoding
# ----------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

# ----------------------
# StintTransformer
# ----------------------
class StintTransformer(nn.Module):
    def __init__(self, n_drivers, n_teams, n_tyres, n_circuits=0, n_modes=3,
                 driver_emb_dim=8, team_emb_dim=8, tyre_emb_dim=4, circuit_emb_dim=12,
                 n_cont_features=10, d_model=64, nhead=4, num_layers=2,
                 dim_feedforward=128, dropout=0.3):
        super().__init__()
        # Only create embeddings if we have classes to embed
        self.driver_emb = nn.Embedding(n_drivers, driver_emb_dim) if n_drivers > 0 else None
        self.team_emb = nn.Embedding(n_teams, team_emb_dim) if n_teams > 0 else None
        self.tyre_emb = nn.Embedding(n_tyres, tyre_emb_dim) if n_tyres > 0 else None
        self.circuit_emb = nn.Embedding(n_circuits, circuit_emb_dim) if n_circuits > 0 else None

        # Adjust input dimension based on which embeddings exist
        actual_driver_dim = driver_emb_dim if n_drivers > 0 else 0
        actual_team_dim = team_emb_dim if n_teams > 0 else 0
        actual_tyre_dim = tyre_emb_dim if n_tyres > 0 else 0
        actual_circuit_dim = circuit_emb_dim if n_circuits > 0 else 0
        input_dim = n_cont_features + actual_driver_dim + actual_team_dim + actual_tyre_dim + actual_circuit_dim 
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   dropout=dropout,
                                                   batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_layer = nn.Linear(d_model, 1)

    def forward(self, cont_feats, driver_idx, team_idx, tyre_idx, circuit_idx=None, mask=None):
        batch_size, seq_len, _ = cont_feats.size()
        # Collect embeddings only for features that exist
        embeddings = [cont_feats]
        
        if self.driver_emb is not None:
            driver_emb = self.driver_emb(driver_idx).unsqueeze(1).repeat(1, seq_len, 1)
            embeddings.append(driver_emb)
        
        if self.team_emb is not None:
            team_emb = self.team_emb(team_idx).unsqueeze(1).repeat(1, seq_len, 1)
            embeddings.append(team_emb)
        
        if self.tyre_emb is not None:
            tyre_emb = self.tyre_emb(tyre_idx)
            embeddings.append(tyre_emb)
        
        if self.circuit_emb is not None and circuit_idx is not None:
            circuit_emb = self.circuit_emb(circuit_idx).unsqueeze(1).repeat(1, seq_len, 1)
            embeddings.append(circuit_emb)
        
        x = torch.cat(embeddings, dim=-1)
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        key_padding_mask = mask if mask is not None else None
        x = self.transformer(x, src_key_padding_mask=key_padding_mask)
        out = self.output_layer(x).squeeze(-1)
        return out

In [0]:
class StintGRU(nn.Module):
    """
    GRU variant (similar to LSTM but with simpler gating mechanism).
    Often faster than LSTM with similar performance.
    """
    def __init__(self, n_drivers, n_teams, n_tyres, n_circuits=0,
                 driver_emb_dim=8, team_emb_dim=8, tyre_emb_dim=4, circuit_emb_dim=12, mode_emb_dim=2,
                 n_cont_features=10, hidden_dim=64, num_layers=2, dropout=0.3):
        super().__init__()
        
        # Only create embeddings if we have classes to embed
        self.driver_emb = nn.Embedding(n_drivers, driver_emb_dim) if n_drivers > 0 else None
        self.team_emb = nn.Embedding(n_teams, team_emb_dim) if n_teams > 0 else None
        self.tyre_emb = nn.Embedding(n_tyres, tyre_emb_dim) if n_tyres > 0 else None
        self.circuit_emb = nn.Embedding(n_circuits, circuit_emb_dim) if n_circuits > 0 else None
        
        # Adjust input dimension based on which embeddings are present
        actual_driver_dim = driver_emb_dim if n_drivers > 0 else 0
        actual_team_dim = team_emb_dim if n_teams > 0 else 0
        actual_tyre_dim = tyre_emb_dim if n_tyres > 0 else 0
        actual_circuit_dim = circuit_emb_dim if n_circuits > 0 else 0
        input_dim = n_cont_features + actual_driver_dim + actual_team_dim + actual_tyre_dim + actual_circuit_dim
        
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        
        self.output_layer = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, cont_feats, driver_idx, team_idx, tyre_idx, circuit_idx=None, mask=None):
        batch_size, seq_len, _ = cont_feats.size()
        
        # Collect embeddings only for features that exist
        embeddings = [cont_feats]
        
        if self.driver_emb is not None:
            driver_emb = self.driver_emb(driver_idx).unsqueeze(1).repeat(1, seq_len, 1)
            embeddings.append(driver_emb)
        
        if self.team_emb is not None:
            team_emb = self.team_emb(team_idx).unsqueeze(1).repeat(1, seq_len, 1)
            embeddings.append(team_emb)
        
        if self.tyre_emb is not None:
            tyre_emb = self.tyre_emb(tyre_idx)
            embeddings.append(tyre_emb)
        
        if self.circuit_emb is not None and circuit_idx is not None:
            circuit_emb = self.circuit_emb(circuit_idx).unsqueeze(1).repeat(1, seq_len, 1)
            embeddings.append(circuit_emb)
        
        x = torch.cat(embeddings, dim=-1)
        
        if mask is not None:
            seq_lengths = (~mask).sum(dim=1).cpu()
            x_packed = pack_padded_sequence(x, seq_lengths, batch_first=True, enforce_sorted=False)
            gru_out_packed, _ = self.gru(x_packed)
            gru_out, _ = pad_packed_sequence(gru_out_packed, batch_first=True)
        else:
            gru_out, _ = self.gru(x)
        
        gru_out = self.dropout(gru_out)
        out = self.output_layer(gru_out).squeeze(-1)
        
        return out

In [0]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

class StintXGBoost:
    """
    XGBoost model for lap time prediction with native categorical support.
    No embeddings needed - XGBoost handles categorical features directly.
    
    Features:
    - Native categorical handling (no one-hot encoding)
    - Efficient memory usage
    - Fast training and inference
    - Handles sequential patterns through lagged features in your data
    """
    def __init__(self, n_estimators=300, max_depth=5, learning_rate=0.05,
                 subsample=0.8, colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=1.0,
                 early_stopping_rounds=50, random_state=42):
        """
        Args:
            n_estimators: Number of boosting rounds
            max_depth: Maximum tree depth
            learning_rate: Step size shrinkage
            subsample: Row sampling ratio per tree
            colsample_bytree: Column sampling ratio per tree
            reg_alpha: L1 regularization
            reg_lambda: L2 regularization
            early_stopping_rounds: Stop if no improvement for N rounds
        """
        self.model = xgb.XGBRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            random_state=random_state,
            early_stopping_rounds=early_stopping_rounds,
            enable_categorical=True,  # Native categorical support
            tree_method='hist',       # Required for categorical features
            max_cat_to_onehot=1,      # Use optimal splits, not one-hot
            objective='reg:squarederror',
            eval_metric='mae'
        )
        
        self.feature_names = None
        # Get categorical features from configuration (instead of hardcoding)
        # Import will be available from %run in training notebook
        try:
            from __main__ import get_xgboost_categorical_features
            self.categorical_features = get_xgboost_categorical_features()
        except (ImportError, AttributeError):
            # Fallback if configuration not loaded
            self.categorical_features = ['Driver_idx', 'Team_idx', 'Compound']
        
    def prepare_data(self, df, categorical_cols=None):
        """
        Prepare DataFrame for XGBoost with categorical features.
        
        Args:
            df: pandas DataFrame with all features
            categorical_cols: List of categorical column names
                            (default: ['Driver', 'Team', 'Compound', 'Mode'])
        
        Returns:
            DataFrame with categorical columns properly typed
        """
        df = df.copy()
        
        if categorical_cols is None:
            categorical_cols = self.categorical_features
        
        # Convert categorical columns to pandas 'category' dtype
        for col in categorical_cols:
            if col in df.columns:
                # XGBoost requires integer or string categorical features
                # Convert floats to integers first
                if df[col].dtype in ['float64', 'float32']:
                    df[col] = df[col].astype('int64')
                df[col] = df[col].astype('category')
        
        return df
    
    def fit(self, X, y, eval_set=None, verbose=True):
        """
        Train the XGBoost model.
        
        Args:
            X: Training features (pandas DataFrame)
            y: Training targets (lap times)
            eval_set: List of (X_val, y_val) tuples for validation
            verbose: Print training progress
        """
        # Store feature names
        self.feature_names = X.columns.tolist()
        
        # Prepare categorical features
        X_prepared = self.prepare_data(X, self.categorical_features)
        
        # Prepare eval_set if provided
        eval_set_prepared = None
        if eval_set is not None:
            eval_set_prepared = [
                (self.prepare_data(X_val, self.categorical_features), y_val)
                for X_val, y_val in eval_set
            ]
        
        # Train
        self.model.fit(
            X_prepared, y,
            eval_set=eval_set_prepared,
            verbose=verbose
        )
        
        return self
    
    def predict(self, X):
        """
        Predict lap times.
        
        Args:
            X: Features (pandas DataFrame)
        
        Returns:
            numpy array of predicted lap times
        """
        X_prepared = self.prepare_data(X, self.categorical_features)
        return self.model.predict(X_prepared)
    
    def get_feature_importance(self, importance_type='gain'):
        """
        Get feature importance scores.
        
        Args:
            importance_type: 'gain', 'weight', or 'cover'
        
        Returns:
            pandas Series of feature importances (sorted)
        """
        importance = self.model.get_booster().get_score(importance_type=importance_type)
        importance_df = pd.Series(importance).sort_values(ascending=False)
        return importance_df
    
    def save(self, path):
        """Save model to file."""
        self.model.save_model(path)
    
    def load(self, path):
        """Load model from file."""
        self.model.load_model(path)
        return self

In [0]:
# ====================================================================
# XGBoost Training: Pre-filter SC/DNF laps
# ====================================================================
# Unlike sequence models, XGBoost treats each lap independently.
# So we FILTER the data upfront rather than masking during training.

import pandas as pd
from pyspark.sql import SparkSession

def prepare_xgboost_data(spark_table_name):
    """
    Load and filter training/validation data for XGBoost.
    
    Key difference from sequence models:
    - XGBoost has no temporal context, so we can't "flow hidden state" through SC laps
    - Therefore, we simply remove invalid laps from training data entirely
    
    Args:
        spark_table_name: Unity Catalog table name (e.g., 'workspace.f1_racing_laptime_pred.raw_training')
    
    Returns:
        pandas DataFrame with only valid laps (green flag + non-DNF)
    """
    spark = SparkSession.builder.getOrCreate()
    
    # Load data from Unity Catalog
    df = spark.table(spark_table_name).toPandas()
    
    print(f"Total laps loaded: {len(df)}")
    
    # Filter to valid laps only
    # Assuming 'status_1' and 'dnf' columns exist in your dataset
    valid_mask = (df['status_1'] == 1) & (df['dnf'] == 0)
    df_filtered = df[valid_mask].copy()
    
    print(f"Valid laps (green flag + non-DNF): {len(df_filtered)}")
    print(f"Removed: {len(df) - len(df_filtered)} invalid laps ({100*(1-len(df_filtered)/len(df)):.1f}%)")
    
    # Drop the mask columns since they're no longer needed
    df_filtered = df_filtered.drop(columns=['status_1', 'dnf'], errors='ignore')
    
    return df_filtered


def train_xgboost_model():
    """
    Complete XGBoost training pipeline with pre-filtered data.
    """
    # Load and filter training data
    train_df = prepare_xgboost_data('workspace.f1_racing_laptime_pred.raw_training')
    val_df = prepare_xgboost_data('workspace.f1_racing_laptime_pred.raw_validating')
    
    # Separate features and target
    target_col = 'LapTime_sec'
    
    # Columns to exclude from features
    exclude_cols = [target_col, 'PitInTime', 'PitOutTime', 'Time', 'LapStartTime', 
                    'Sector1Time', 'Sector2Time', 'Sector3Time', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST']
    feature_cols = [c for c in train_df.columns if c not in exclude_cols]
    
    X_train = train_df[feature_cols]
    y_train = train_df[target_col]
    
    X_val = val_df[feature_cols]
    y_val = val_df[target_col]
    
    print(f"\nTraining features: {len(feature_cols)} columns")
    print(f"Training samples: {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")
    
    # Initialize XGBoost model
    xgb_model = StintXGBoost(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1.0,
        reg_lambda=1.0,
        early_stopping_rounds=50
    )
    
    # Train with validation monitoring
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=True
    )
    
    # Evaluate
    train_preds = xgb_model.predict(X_train)
    val_preds = xgb_model.predict(X_val)
    
    train_mae = np.mean(np.abs(y_train - train_preds))
    val_mae = np.mean(np.abs(y_val - val_preds))
    
    print(f"\nFinal Results:")
    print(f"Train MAE: {train_mae:.4f}s")
    print(f"Val MAE: {val_mae:.4f}s")
    
    # Feature importance
    print("\nTop 10 Most Important Features:")
    importance = xgb_model.get_feature_importance(importance_type='gain')
    print(importance.head(10))
    
    return xgb_model


# ====================================================================
# Summary: Two Approaches for SC/DNF Handling
# ====================================================================
# 
# SEQUENCE MODELS (LSTM, GRU, Transformer, CNN+LSTM):
#   ✓ Keep SC/DNF laps in sequences (preserve temporal context)
#   ✓ Mask them from loss computation
#   ✓ Hidden states flow through, but no gradient penalty
# 
# XGBoost / Random Forest:
#   ✓ Filter SC/DNF laps before training
#   ✓ No temporal context anyway, so just remove bad data
#   ✓ Cleaner and simpler
# 
# Both achieve the same goal: Don't train on corrupted lap times!

In [0]:
class StintCNNLSTM(nn.Module):
    """
    CNN+LSTM hybrid for lap time prediction.
    
    Architecture:
    1. CNN extracts local temporal patterns (3-5 lap windows)
       - Tire degradation curve shapes
       - Post-pit recovery patterns
       - Short-term pace trends
    
    2. LSTM processes CNN features to capture long-term dependencies
       - Fuel load decrease over stint
       - Cumulative tire wear
       - Stint-level strategy impact
    
    Why this works:
    - CNN excels at detecting local patterns (like "0.3s/lap degradation")
    - LSTM excels at remembering long-term context (like "15 laps into stint")
    - Each component does what it's best at!
    """
    def __init__(self, n_drivers, n_teams, n_tyres, n_circuits=0, n_modes=3,
                 driver_emb_dim=8, team_emb_dim=8, tyre_emb_dim=4, circuit_emb_dim=12,
                 n_cont_features=10, cnn_channels=64, cnn_kernel_size=3,
                 lstm_hidden=64, lstm_layers=2, dropout=0.3):
        super().__init__()
        
        # Embeddings (only create if we have classes to embed)
        self.driver_emb = nn.Embedding(n_drivers, driver_emb_dim) if n_drivers > 0 else None
        self.team_emb = nn.Embedding(n_teams, team_emb_dim) if n_teams > 0 else None
        self.tyre_emb = nn.Embedding(n_tyres, tyre_emb_dim) if n_tyres > 0 else None
        self.circuit_emb = nn.Embedding(n_circuits, circuit_emb_dim) if n_circuits > 0 else None
        
        # Total input dimension (adjust based on which embeddings exist)
        actual_driver_dim = driver_emb_dim if n_drivers > 0 else 0
        actual_team_dim = team_emb_dim if n_teams > 0 else 0
        actual_tyre_dim = tyre_emb_dim if n_tyres > 0 else 0
        actual_circuit_dim = circuit_emb_dim if n_circuits > 0 else 0
        input_dim = n_cont_features + actual_driver_dim + actual_team_dim + actual_tyre_dim + actual_circuit_dim 
        
        # ==========================================
        # CNN Layers: Extract Local Patterns
        # ==========================================
        # Conv1D operates on (batch, channels, seq_len)
        # We treat features as "channels" and sequence as "width"
        
        self.conv1 = nn.Conv1d(
            in_channels=input_dim,
            out_channels=cnn_channels,
            kernel_size=cnn_kernel_size,
            padding=cnn_kernel_size // 2  # Same padding to preserve sequence length
        )
        self.conv2 = nn.Conv1d(
            in_channels=cnn_channels,
            out_channels=cnn_channels,
            kernel_size=cnn_kernel_size,
            padding=cnn_kernel_size // 2
        )
        
        self.conv_dropout = nn.Dropout(dropout)
        self.batch_norm1 = nn.BatchNorm1d(cnn_channels)
        self.batch_norm2 = nn.BatchNorm1d(cnn_channels)
        
        # ==========================================
        # LSTM Layers: Capture Long-term Dependencies
        # ==========================================
        self.lstm = nn.LSTM(
            input_size=cnn_channels,  # Takes CNN output as input
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0,
            bidirectional=False
        )
        
        self.lstm_dropout = nn.Dropout(dropout)
        
        # Output layer
        self.output_layer = nn.Linear(lstm_hidden, 1)
        
    def forward(self, cont_feats, driver_idx, team_idx, tyre_idx, circuit_idx=None, mask=None):
        """
        Args:
            cont_feats: (batch_size, seq_len, n_cont_features)
            driver_idx: (batch_size,)
            team_idx: (batch_size,)
            tyre_idx: (batch_size, seq_len)
            circuit_idx: (batch_size,) - circuit index
            mask: (batch_size, seq_len) - True for padded positions
        
        Returns:
            out: (batch_size, seq_len) - predicted lap times
        """
        batch_size, seq_len, _ = cont_feats.size()
        
        # Collect embeddings only for features that exist
        embeddings = [cont_feats]
        
        if self.driver_emb is not None:
            driver_emb = self.driver_emb(driver_idx).unsqueeze(1).repeat(1, seq_len, 1)
            embeddings.append(driver_emb)
        
        if self.team_emb is not None:
            team_emb = self.team_emb(team_idx).unsqueeze(1).repeat(1, seq_len, 1)
            embeddings.append(team_emb)
        
        if self.tyre_emb is not None:
            tyre_emb = self.tyre_emb(tyre_idx)
            embeddings.append(tyre_emb)
        
        if self.circuit_emb is not None and circuit_idx is not None:
            circuit_emb = self.circuit_emb(circuit_idx).unsqueeze(1).repeat(1, seq_len, 1)
            embeddings.append(circuit_emb)
        
        # Concatenate all features: (batch_size, seq_len, input_dim)
        x = torch.cat(embeddings, dim=-1)
        
        # ==========================================
        # Step 1: CNN extracts local patterns
        # ==========================================
        # Conv1D expects (batch, channels, seq_len)
        x = x.permute(0, 2, 1)  # (batch_size, input_dim, seq_len)
        
        # First conv layer
        x = self.conv1(x)
        x = self.batch_norm1(x)
        x = torch.relu(x)
        x = self.conv_dropout(x)
        
        # Second conv layer
        x = self.conv2(x)
        x = self.batch_norm2(x)
        x = torch.relu(x)
        x = self.conv_dropout(x)
        
        # Permute back to (batch_size, seq_len, cnn_channels) for LSTM
        x = x.permute(0, 2, 1)
        
        # ==========================================
        # Step 2: LSTM processes CNN features
        # ==========================================
        if mask is not None:
            # Pack sequences for efficient processing
            seq_lengths = (~mask).sum(dim=1).cpu()
            x_packed = pack_padded_sequence(
                x, 
                seq_lengths, 
                batch_first=True, 
                enforce_sorted=False
            )
            lstm_out_packed, _ = self.lstm(x_packed)
            lstm_out, _ = pad_packed_sequence(lstm_out_packed, batch_first=True)
        else:
            lstm_out, _ = self.lstm(x)
        
        # Apply dropout and output layer
        lstm_out = self.lstm_dropout(lstm_out)
        out = self.output_layer(lstm_out).squeeze(-1)  # (batch_size, seq_len)
        
        return out


# ==========================================
# Usage Example:
# ==========================================
# model = StintCNNLSTM(
#     n_drivers=20,
#     n_teams=10,
#     n_tyres=5,
#     n_modes=3,
#     n_cont_features=10,
#     cnn_channels=64,      # CNN extracts 64 feature maps
#     cnn_kernel_size=3,    # Looks at 3-lap windows
#     lstm_hidden=64,       # LSTM hidden state size
#     lstm_layers=2,        # 2 LSTM layers
#     dropout=0.3
# )
#
# What happens during forward pass:
# 1. Input: [Lap1_features, Lap2_features, ..., Lap20_features]
# 2. CNN sees 3-lap windows:
#    - [Lap1, Lap2, Lap3] → learns "early stint behavior"
#    - [Lap2, Lap3, Lap4] → learns "degradation rate"
#    - [Lap18, Lap19, Lap20] → learns "late stint behavior"
# 3. LSTM sees CNN outputs:
#    - Remembers "stint started 20 laps ago"
#    - Remembers "traffic at lap 5"
#    - Combines local patterns + long-term context
# 4. Output: Predicted lap time using both!

In [0]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np

def train_epoch_with_masking(model, dataloader, criterion, optimizer, device):
    """
    Training loop with loss masking for Safety Car and DNF laps.
    
    Key insight:
    - SC and DNF laps remain in the sequence (hidden state flows through)
    - But we only compute loss on VALID laps (green flag + non-DNF)
    - This preserves temporal continuity without corrupting the loss
    
    Args:
        model: StintLSTM, StintGRU, StintTransformer, or StintCNNLSTM
        dataloader: Returns batches with keys:
                   ['cont_feats', 'driver_idx', 'team_idx', 'tyre_idx', 
                    'mode_idx', 'mask', 'LapTime_sec', 'status_1', 'dnf']
        criterion: Loss function (e.g., nn.MSELoss(reduction='none'))
        optimizer: torch.optim optimizer
        device: 'cuda' or 'cpu'
    
    Returns:
        avg_loss: Average loss over valid laps only
    """
    model.train()
    total_loss = 0
    total_valid_laps = 0
    
    for batch in dataloader:
        # Move batch to device
        cont_feats = batch['cont_feats'].to(device)        # (batch, seq_len, n_features)
        driver_idx = batch['driver_idx'].to(device)        # (batch,)
        team_idx = batch['team_idx'].to(device)            # (batch,)
        tyre_idx = batch['tyre_idx'].to(device)            # (batch, seq_len)
        mode_idx = batch['mode_idx'].to(device)            # (batch, seq_len)
        mask = batch['mask'].to(device)                    # (batch, seq_len) - True for padding
        
        targets = batch['LapTime_sec'].to(device)          # (batch, seq_len)
        status_1 = batch['status_1'].to(device)            # (batch, seq_len) - 1 for green flag
        dnf = batch['dnf'].to(device)                      # (batch, seq_len) - 0 for valid, 1 for DNF
        
        # Forward pass - model sees ALL laps (including SC/DNF)
        predictions = model(cont_feats, driver_idx, team_idx, tyre_idx, mode_idx, mask)
        # predictions shape: (batch, seq_len)
        
        # ====================================================================
        # KEY MASKING LOGIC: Only compute loss on valid laps
        # ====================================================================
        # Valid lap = green flag (status_1 == 1) AND non-DNF (dnf == 0) AND not padding
        is_valid = (status_1 == 1) & (dnf == 0) & (~mask)
        
        # Compute loss only on valid laps
        # Note: criterion must have reduction='none' to get per-element losses
        loss_per_lap = criterion(predictions, targets)  # (batch, seq_len)
        
        # Apply mask: zero out losses for invalid laps
        masked_loss = loss_per_lap * is_valid.float()
        
        # Average over valid laps only
        num_valid = is_valid.sum()
        if num_valid > 0:
            loss = masked_loss.sum() / num_valid
        else:
            loss = masked_loss.sum()  # Fallback if no valid laps (rare)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Optional: Gradient clipping (helps with sequence models)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        # Accumulate stats
        total_loss += loss.item() * num_valid.item()
        total_valid_laps += num_valid.item()
    
    avg_loss = total_loss / total_valid_laps if total_valid_laps > 0 else 0
    return avg_loss


def validate_with_masking(model, dataloader, criterion, device):
    """
    Validation loop with same masking logic.
    """
    model.eval()
    total_loss = 0
    total_valid_laps = 0
    
    with torch.no_grad():
        for batch in dataloader:
            cont_feats = batch['cont_feats'].to(device)
            driver_idx = batch['driver_idx'].to(device)
            team_idx = batch['team_idx'].to(device)
            tyre_idx = batch['tyre_idx'].to(device)
            mode_idx = batch['mode_idx'].to(device)
            mask = batch['mask'].to(device)
            
            targets = batch['LapTime_sec'].to(device)
            status_1 = batch['status_1'].to(device)
            dnf = batch['dnf'].to(device)
            
            # Forward pass
            predictions = model(cont_feats, driver_idx, team_idx, tyre_idx, mode_idx, mask)
            
            # Mask invalid laps
            is_valid = (status_1 == 1) & (dnf == 0) & (~mask)
            
            # Compute masked loss
            loss_per_lap = criterion(predictions, targets)
            masked_loss = loss_per_lap * is_valid.float()
            
            num_valid = is_valid.sum()
            if num_valid > 0:
                loss = masked_loss.sum() / num_valid
            else:
                loss = masked_loss.sum()
            
            total_loss += loss.item() * num_valid.item()
            total_valid_laps += num_valid.item()
    
    avg_loss = total_loss / total_valid_laps if total_valid_laps > 0 else 0
    return avg_loss


# ====================================================================
# Full Training Example
# ====================================================================
def train_sequence_model():
    """
    Complete training example using loss masking.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Initialize model (choose one)
    model = StintLSTM(
        n_drivers=20,
        n_teams=10,
        n_tyres=5,
        n_modes=3,
        n_cont_features=30,  # Adjust based on your actual feature count
        hidden_dim=64,
        num_layers=2,
        dropout=0.3
    ).to(device)
    
    # CRITICAL: Use reduction='none' to get per-lap losses
    criterion = nn.MSELoss(reduction='none')
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    # Assuming you have train_dataloader and val_dataloader
    # train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    # val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    
    num_epochs = 50
    best_val_loss = float('inf')
    
    for epoch in range(num_epochs):
        # Train with masking
        train_loss = train_epoch_with_masking(model, train_dataloader, criterion, optimizer, device)
        
        # Validate with masking
        val_loss = validate_with_masking(model, val_dataloader, criterion, device)
        
        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_model.pth')
            print(f"  → Saved new best model (val_loss: {val_loss:.4f})")
    
    return model


# ====================================================================
# Why This Works:
# ====================================================================
# ✓ SC laps stay in sequence → LSTM hidden state flows through them
# ✓ Model learns "SC happened at lap 15" as context
# ✓ But we don't penalize the model for predicting corrupted SC lap times
# ✓ Same logic for DNF laps - model sees them but isn't penalized
# 
# Result: Model gets full temporal context without learning from bad data